# Supermarket Sales Analysis with AI
### AICTE | IBM SkillsBuild Data Analytics with AI Academic Internship Program
**Submitted by:** Dasarla Vinayak

This project performs data cleaning, exploratory data analysis, visualization, business insight extraction, and a small AI component using K-Means clustering for transaction segmentation.

## 1. Problem Statement
Analyze supermarket transaction data to understand sales performance across branches, categories, products, customers, payment methods, and time periods. The project also applies an unsupervised machine-learning technique to segment transactions by numerical purchasing characteristics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.titlesize'] = 14


In [ ]:
# Load dataset
file_path = 'supermarket_sales_500_rows.csv'
df = pd.read_csv(file_path)
df.head()


In [ ]:
print('Shape:', df.shape)
print('Columns:', list(df.columns))
print('\nData types:')
print(df.dtypes)
print('\nMissing values:')
print(df.isna().sum())
print('\nDuplicate invoice IDs:', df['Invoice ID'].duplicated().sum())


## 2. Data Cleaning and Validation

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['Sales_Calculated'] = df['Quantity'] * df['Unit Price']
df['Sales_Check'] = np.isclose(df['Sales'], df['Sales_Calculated'], atol=0.01)

print('Missing values after date conversion:', df.isna().sum().sum())
print('Rows where Sales != Quantity × Unit Price:', (~df['Sales_Check']).sum())
print('Date range:', df['Date'].min().date(), 'to', df['Date'].max().date())


## 3. Descriptive Statistics

In [ ]:
df[['Quantity', 'Unit Price', 'Rating', 'Sales']].describe().round(2)


## 4. Exploratory Data Analysis

In [ ]:
branch_sales = df.groupby(['Branch', 'City'])['Sales'].agg(['sum', 'count', 'mean']).sort_values('sum', ascending=False)
branch_sales.round(2)


In [ ]:
category_sales = df.groupby('Category')['Sales'].sum().sort_values(ascending=False)
category_sales.round(2)


In [ ]:
product_sales = df.groupby('Product')['Sales'].sum().sort_values(ascending=False)
product_sales.head(10).round(2)


In [ ]:
payment_counts = df['Payment'].value_counts()
payment_counts


In [ ]:
customer_summary = df.groupby('Customer Type')['Sales'].agg(['sum', 'count', 'mean']).round(2)
customer_summary


In [ ]:
monthly_sales = df.groupby(df['Date'].dt.to_period('M'))['Sales'].sum()
monthly_sales.round(2)


In [ ]:
fig, ax = plt.subplots()
branch_sales['sum'].plot(kind='bar', ax=ax)
ax.set_title('Total Sales by Branch / City')
ax.set_ylabel('Sales (₹)')
plt.xticks(rotation=0)
plt.show()


In [ ]:
fig, ax = plt.subplots()
category_sales.plot(kind='bar', ax=ax)
ax.set_title('Sales by Product Category')
ax.set_ylabel('Sales (₹)')
plt.xticks(rotation=35, ha='right')
plt.show()


In [ ]:
fig, ax = plt.subplots()
top = product_sales.head(10).sort_values()
top.plot(kind='barh', ax=ax)
ax.set_title('Top 10 Products by Sales')
ax.set_xlabel('Sales (₹)')
plt.show()


In [ ]:
fig, ax = plt.subplots()
payment_counts.plot(kind='bar', ax=ax)
ax.set_title('Transactions by Payment Method')
ax.set_ylabel('Transactions')
plt.xticks(rotation=0)
plt.show()


In [ ]:
fig, ax = plt.subplots()
monthly_sales.plot(marker='o', ax=ax)
ax.set_title('Monthly Sales Trend')
ax.set_ylabel('Sales (₹)')
plt.xticks(rotation=35)
plt.show()


## 5. AI Component — K-Means Transaction Segmentation
Because the dataset does not contain a persistent customer ID, the AI component segments **transactions**, not individual customers. The model uses Quantity, Unit Price, Rating, and Sales. Features are standardized before K-Means clustering.

In [ ]:
features = ['Quantity', 'Unit Price', 'Rating', 'Sales']
X = StandardScaler().fit_transform(df[features])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['Transaction_Segment'] = kmeans.fit_predict(X) + 1

segment_summary = df.groupby('Transaction_Segment')[features].mean().round(2)
segment_summary


In [ ]:
# Give the clusters business-friendly labels based on average sales
segment_order = segment_summary['Sales'].sort_values().index.tolist()
segment_labels = {
    segment_order[0]: 'Lower-Value Transactions',
    segment_order[1]: 'Mid-Value Transactions',
    segment_order[2]: 'Higher-Value Transactions'
}
df['Transaction_Segment_Label'] = df['Transaction_Segment'].map(segment_labels)

df.groupby('Transaction_Segment_Label')[features].mean().round(2)


In [ ]:
segment_sales = df.groupby('Transaction_Segment_Label')['Sales'].mean().sort_values()
fig, ax = plt.subplots()
segment_sales.plot(kind='bar', ax=ax)
ax.set_title('AI Transaction Segmentation: Average Sales')
ax.set_ylabel('Average Transaction Sales (₹)')
plt.xticks(rotation=20, ha='right')
plt.show()


## 6. Key Business Insights
- Branch C in Mumbai records the highest total sales.
- Beverages is the highest-sales category.
- Cheese is the highest-sales individual product.
- UPI is the most frequently used payment method, although all four payment methods are relatively close.
- Member transactions generate more total revenue because there are more member transactions, while the average Normal transaction is slightly higher.
- The AI clustering groups transactions into lower-, mid-, and higher-value segments based on numerical purchasing behavior.

## 7. Conclusion
The analysis converts raw supermarket transactions into measurable business insights. The AI segmentation adds an exploratory machine-learning layer that can support differentiated promotions and transaction-level analysis. The dataset is limited to 500 transactions, so the results should be treated as descriptive for this sample rather than as a complete representation of supermarket behavior.